# Combinatorial CoWork 2026 — Notebook 02: Resolutions, `Hom`, `Ext`, and `Tor`

Audience:
- Participants who want the most commutative-algebra-flavored part of the library first: common encodings, resolutions, tables, and derived functors.

Learning goals:
- Build two tiny rectangle-indicator modules in `Z^2` through the flange interface.
- Common-encode them and compute `Hom`, `Ext`, `Tor`, projective resolutions, and injective resolutions.
- Use `betti_table(...)`, `bass_table(...)`, `degree_range(...)`, and `dim(...)` as the cheap-first inspection surfaces.


In [1]:
NOTEBOOK_STEM = "02_resolutions_hom_ext_tor"

_TO_ROOT = let
    dir = abspath(pwd())
    root = nothing
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "TamerOp.jl"))
            root = dir
            break
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repo root containing Project.toml and src/TamerOp.jl from pwd()=$(pwd()).")
        dir = parent
    end
    root
end

import Pkg
Pkg.activate(_TO_ROOT; io=devnull)

if !isdefined(Main, :TamerOp)
    @eval Main using TamerOp
end

TO = Main.TamerOp
CM = TO.CoreModules
OPT = TO.Options
sc = TO.SessionCache()
outdir = joinpath(_TO_ROOT, "examples", "_outputs", "combinatorial_cowork_2026", NOTEBOOK_STEM)
mkpath(outdir)


"/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/02_resolutions_hom_ext_tor"

## 1. Build two rectangle-indicator flanges

We use `SyntheticData.orthant_bar_flange(...)` directly here rather than a notebook-local helper. The point is that the source object is already a mathematically meaningful algebraic presentation, but we still want the same workflow habit: build the source, inspect it, then encode it.


In [2]:
FM = TO.SyntheticData.orthant_bar_flange(
    bars=[([0, 0], [2, 2])],
    field=CM.QQField(),
)

FN = TO.SyntheticData.orthant_bar_flange(
    bars=[([1, 1], [3, 3])],
    field=CM.QQField(),
)

enc_opts = OPT.EncodingOptions(backend=:zn, max_regions=50_000, poset_kind=:signature)


EncodingOptions
  backend: :zn
  max_regions: 50000
  strict_eps: nothing
  poset_kind: :signature
  field: TamerOp.CoreModules.CoeffFields.QQField()

Before encoding, inspect the two source objects.

At this level, the important question is still “what algebraic data did we build?” rather than “what query do we want to run next?”

The summaries here are source-object summaries. They tell us about the flange presentations themselves, not yet about a common encoding.


In [3]:
(; FM=TO.describe(FM),
   FN=TO.describe(FN),
   enc_opts=enc_opts)


(FM = (kind = :flange, ambient_dim = 2, field = "TamerOp.CoreModules.CoeffFields.QQField", nflats = 1, ninjectives = 1, matrix_size = (1, 1)), FN = (kind = :flange, ambient_dim = 2, field = "TamerOp.CoreModules.CoeffFields.QQField", nflats = 1, ninjectives = 1, matrix_size = (1, 1)), enc_opts = EncodingOptions(backend=:zn, poset_kind=:signature, max_regions=50000))

## 2. Common-encode the pair

`encode((FM, FN), enc_opts; ...)` is the notebook-facing path when two modules should be compared or combined on one common encoding.

This step matters because `Hom` and `Ext` are most natural when the two modules live on the same encoded finite poset with the same classifier map. We also inspect one explicit geometry summary so the common encoding is visible as geometry, not just as an abstract contract.


In [4]:
encM, encN = TO.encode((FM, FN), enc_opts; cache=sc)
regions_spec = TO.Visualization.visual_spec(TO.encoding_map(encM); kind=:regions)


VisualizationSpec
  visual_kind: regions
  title: "Zn encoding"
  subtitle: ""
  nlayers: 1
  npanels: 0
  layer_types: (:RectLayer,)
  axes: (xlabel = "g1", ylabel = "g2", xlimits = (-1.0, 4.0), ylimits = (-1.0, 1.0), zlabel = "z", zlimits = nothing, aspect = :equal, xticks = nothing, yticks = nothing)
  metadata: (figure_size = (760, 620), legend_position = :none, object = :zn_encoding_map, nregions = 9, query_count = 0)

Now inspect the encoded pair.

The two key boolean checks are:

- `shared_poset`: do the encoded modules land on the same finite poset?
- `shared_classifier`: do they reuse the same encoding map `pi`?

For this notebook, we want both answers to be `true`. That is what makes the later algebraic comparison cells direct and readable.


In [5]:
(; M_summary=TO.describe(encM),
   N_summary=TO.describe(encN),
   shared_poset=(TO.encoding_poset(encM) === TO.encoding_poset(encN)),
   shared_classifier=(TO.encoding_map(encM) === TO.encoding_map(encN)),
   regions_visual_summary=TO.Visualization.visual_summary(regions_spec))


(M_summary = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Int64}, Vector{Int64}}, Vector{Tuple{Int64, Int64}}, TamerOp.CoreModules.EncodingCache}, compiled = true, backend = :zn, has_cohomology = true, has_presentation = true, module_dims = [0, 0, 0, 0, 0, 1, 1, 0, 0]), N_summary = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncodi

## 3. Projective and injective diagnostics

These are the cheapest standard commutative-algebra summaries to show before diving into full derived-functor objects.

The workflow object returned by `resolve(...)` is a typed result wrapper. We ask for:

- one projective resolution of `encM`,
- one injective resolution of `encN`.

Then we immediately inspect the cheap table-valued summaries rather than materializing more structure than the notebook needs.


In [6]:
res_opts = OPT.ResolutionOptions(maxlen=2, minimal=true, check=true)
res_proj = TO.resolve(encM; kind=:projective, opts=res_opts, cache=sc)
res_inj = TO.resolve(encN; kind=:injective, opts=res_opts, cache=sc)


ResolutionResult
  resolution_type: TamerOp.DerivedFunctors.Resolutions.InjectiveResolution{Rational{BigInt}}
  source_type: EncodingResult{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Int64}, Vector{Int64}}, Vector{Tuple{Int64, Int64}}, TamerOp.CoreModules.EncodingCache}, TamerOp.FiniteFringe.FringeModule{Rational{BigInt}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}}, TamerOp.FlangeZn.Flange{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, 2}, @NamedTuple{}}
  has_betti: false
  has_minimality: false
  opts_type: TamerOp.Options.ResolutionOptions

`result_summary(...)` is the owner-local inspection helper for these resolution results. The two tables are the standard cheap-first summaries:

- `betti_table(...)` for the projective side,
- `bass_table(...)` for the injective side.

This is the first place in the notebook where the audience can see a familiar algebraic summary before any explicit `Ext` or `Tor` object appears.


In [7]:
(; projective=TO.result_summary(res_proj),
   injective=TO.result_summary(res_inj),
   betti=TO.betti_table(res_proj),
   bass=TO.bass_table(res_inj))


(projective = (kind = :resolution_result, resolution_type = TamerOp.DerivedFunctors.Resolutions.ProjectiveResolution{Rational{BigInt}}, source_type = EncodingResult{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.EncodingCore.CompiledEncoding{TamerOp.ZnEncoding.ZnEncodingMap{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Int64}, Vector{Int64}}, Vector{Tuple{Int64, Int64}}, TamerOp.CoreModules.EncodingCache}, TamerOp.FiniteFringe.FringeModule{Rational{BigInt}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}}, TamerOp.FlangeZn.Flange{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, 2}, @NamedTuple{}}, has_betti = true, has_minimality = false, opts_type = TamerOp.Options.ResolutionOptions), injective = (kind = :resolution_result, resolution_type = TamerOp

## 4. `Hom` and `Ext`

For the shared-encoding pair, the next step is to ask for the basic comparison objects.

- `hom_dimension(...)` is the cheapest scalar summary.
- `hom(...)` returns the actual `Hom` object.
- `ext(...)` returns the graded `Ext` object.

The important notebook habit is to inspect the cheap scalar and degree summaries first, then only later touch representatives or coordinates.


In [8]:
hom_dim = TO.hom_dimension(encM, encN; cache=sc)
H = TO.hom(encM, encN; cache=sc)
E = TO.ext(encM, encN; maxdeg=2, cache=sc)


ExtSpaceProjective
  field: TamerOp.CoreModules.CoeffFields.QQField()
  nvertices: 9
  degree_range: 0:2
  nonzero_degrees: ()
  degree_dimensions: Dict{Int64, Int64}()
  total_dimension: 0

Here the two main cheap query surfaces are:

- `degree_range(E)`: which cohomological degrees are present?
- `dim(E, t)`: what is the dimension in degree `t`?

This keeps the graded object notebook-friendly. Users do not need a basis or representative cycle just to answer the first structural questions.


In [9]:
ext_degrees = collect(TO.Advanced.degree_range(E))
ext_dims = [TO.Advanced.dim(E, t) for t in TO.Advanced.degree_range(E)]

(; hom_dim=hom_dim,
   hom_summary=TO.describe(H),
   ext_summary=TO.describe(E),
   ext_degrees=ext_degrees,
   ext_dims=ext_dims)


(hom_dim = 0, hom_summary = (kind = :hom_space, field = TamerOp.CoreModules.CoeffFields.QQField(), nvertices = 9, dimension = 0, degree_range = 0:0, basis_cached = false, source_total_dim = 2, target_total_dim = 2), ext_summary = (kind = :ext_space, model = :projective, field = TamerOp.CoreModules.CoeffFields.QQField(), nvertices = 9, degree_range = 0:2, nonzero_degrees = (), degree_dimensions = Dict{Int64, Int64}(), total_dimension = 0), ext_degrees = [0, 1, 2], ext_dims = [0, 0, 0])

## 5. `Tor` needs an opposite-poset pair

`Tor` is not shown on the shared-encoding pair because its variance is mathematically important: we need a right-module/left-module setup on opposite posets.

So this cell is intentionally explicit. It is a tiny finite-poset construction that keeps the contract honest rather than pretending `Tor` works by the same simple path as `Hom` and `Ext`.


In [10]:
TOA = TO.Advanced
P = TOA.FinitePoset(Bool[1 1; 0 1]; check=false)
Pop = TOA.FinitePoset(Bool[1 0; 1 1]; check=false)
idP = TOA.EncodingMap(P, P, [1, 2])
idPop = TOA.EncodingMap(Pop, Pop, [1, 2])
field = CM.QQField()
HL = TOA.one_by_one_fringe(P, TOA.principal_upset(P, 1), TOA.principal_downset(P, 2), TO.QQ(1); field=field)
HRop = TOA.one_by_one_fringe(Pop, TOA.principal_upset(Pop, 2), TOA.principal_downset(Pop, 1), TO.QQ(1); field=field)
encL = TO.EncodingResult(P, TOA.pmodule_from_fringe(HL), idP; H=HL, opts=OPT.EncodingOptions(field=field), backend=:workshop)
encRop = TO.EncodingResult(Pop, TOA.pmodule_from_fringe(HRop), idPop; H=HRop, opts=OPT.EncodingOptions(field=field), backend=:workshop)
T = TO.tor(encRop, encL; maxdeg=2, cache=sc)


TorSpace
  model: first
  field: TamerOp.CoreModules.CoeffFields.QQField()
  nvertices: 2
  degree_range: 0:2
  nonzero_degrees: (0,)
  degree_dimensions: Dict(0 => 1)
  total_dimension: 1

Again we stay with cheap graded summaries first.

The pattern is the same as for `Ext`:

- ask for `degree_range(T)`,
- then ask for `dim(T, t)` degree by degree,
- and keep the typed result object itself visible through `describe(...)`.


In [11]:
tor_degrees = collect(TO.Advanced.degree_range(T))
tor_dims = [TO.Advanced.dim(T, t) for t in TO.Advanced.degree_range(T)]

(; tor_degrees=tor_degrees,
   tor_dims=tor_dims,
   tor_summary=TO.describe(T))


(tor_degrees = [0, 1, 2], tor_dims = [1, 0, 0], tor_summary = (kind = :tor_space, model = :first, field = TamerOp.CoreModules.CoeffFields.QQField(), nvertices = 2, degree_range = 0:2, nonzero_degrees = (0,), degree_dimensions = Dict(0 => 1), total_dimension = 1))

## 6. One coordinates/representative round-trip for `Ext`

Only after the cheap degree summary do we touch a representative.

If a degree is nonzero, we:

1. build a coordinate vector in that degree,
2. ask `representative(...)` for a concrete class,
3. recover coordinates with `coordinates(...)`.

This is the smallest honest example of the basis/representative round-trip.


In [12]:
deg = first(TO.Advanced.degree_range(E))
deg_dim = TO.Advanced.dim(E, deg)
if deg_dim > 0
    coeffs = zeros(TO.QQ, deg_dim)
    coeffs[1] = TO.QQ(1)
    z = TO.Advanced.representative(E, deg, coeffs)
    recovered = TO.Advanced.coordinates(E, deg, z)
    (; degree=deg,
       degree_dim=deg_dim,
       input_coeffs=coeffs,
       recovered_coeffs=recovered,
       exact_round_trip=(recovered == coeffs))
else
    "No nonzero Ext degree was available for the round-trip cell."
end


"No nonzero Ext degree was available for the round-trip cell."

## 7. Export one geometry summary and one rank summary

For slides or later reference, keep the exported artifacts minimal.

We export:

- one regions view for the common encoding geometry,
- one rank view for `encM`.

This keeps the output aligned with the algebraic story: geometry of the shared encoding plus one cheap invariant summary.


In [13]:
rank_M = TO.rank_invariant(encM; opts=OPT.InvariantOptions(threads=false), cache=sc)
exports = TO.save_visuals(
    outdir,
    [
        (; stem="rectangles_regions", obj=TO.encoding_map(encM), kind=:regions),
        (; stem="rectangles_rank_heatmap", obj=rank_M, kind=:rank_heatmap),
    ];
    format=:html,
    backend=:cairomakie,
)

Dict(TO.export_stem(r) => TO.export_path(r) for r in exports)


Dict{String, String} with 2 entries:
  "rectangles_rank_heatmap" => "/home/eriknovak/Documents/duke_fall_2025/tamer-…
  "rectangles_regions"      => "/home/eriknovak/Documents/duke_fall_2025/tamer-…

## Try this next

Shift the second rectangle farther away, then rerun the common-encoding and derived-functor cells.

The question to watch is not just whether the objects still exist, but how the cheap summaries change:

- `hom_dimension(...)`,
- `ext_dims`,
- and, if you rebuild a comparable `Tor` setup, the `tor_dims` as well.


In [14]:
FN_shifted = TO.SyntheticData.orthant_bar_flange(
    bars=[([2, 2], [4, 4])],
    field=CM.QQField(),
)

encM2, encN2 = TO.encode((FM, FN_shifted), enc_opts; cache=sc)
E2 = TO.ext(encM2, encN2; maxdeg=2, cache=sc)

(; shifted_hom_dim=TO.hom_dimension(encM2, encN2; cache=sc),
   shifted_ext_degrees=collect(TO.Advanced.degree_range(E2)),
   shifted_ext_dims=[TO.Advanced.dim(E2, t) for t in TO.Advanced.degree_range(E2)])


(shifted_hom_dim = 0, shifted_ext_degrees = [0, 1, 2], shifted_ext_dims = [0, 0, 0])